# Getting Started with Occulus

Occulus is a multi-platform point cloud analysis library for aerial (ALS), terrestrial (TLS), and UAV LiDAR.

This notebook walks through the core concepts:
- Creating and inspecting `PointCloud` objects
- Reading and writing LAS/XYZ files
- Basic filtering and statistics
- Platform-aware subclasses (`AerialCloud`, `TerrestrialCloud`, `UAVCloud`)

In [ ]:
import numpy as np
import occulus
print('occulus version:', occulus.__version__)

## 1. Creating a PointCloud

In [ ]:
from occulus.types import PointCloud, AerialCloud, TerrestrialCloud, UAVCloud, Platform

rng = np.random.default_rng(42)
xyz = rng.standard_normal((10_000, 3))

cloud = PointCloud(xyz)
print(cloud)
print(f'Points   : {cloud.n_points:,}')
print(f'Platform : {cloud.platform}')
print(f'Bounds   :\n{cloud.bounds}')
print(f'Centroid : {cloud.centroid}')

## 2. Platform-Aware Subclasses

Occulus tracks the acquisition platform. Aerial clouds have extra methods for
ASPRS classification; terrestrial clouds carry scan positions.

In [ ]:
# Aerial cloud with ASPRS classification
classification = np.zeros(10_000, dtype=np.uint8)
classification[:3000] = 2  # ground
classification[3000:6000] = 4  # medium vegetation
classification[6000:] = 6  # building

aerial = AerialCloud(xyz, classification=classification)
ground_mask = aerial.ground_points()
print(f'Ground points : {ground_mask.sum():,}')

# UAV cloud
uav = UAVCloud(xyz, is_photogrammetric=True)
print(f'UAV cloud — photogrammetric: {uav.is_photogrammetric}')

## 3. Reading Point Clouds

Occulus reads LAS, LAZ, PLY, PCD, XYZ, and CSV files with a unified `read()` function.

In [ ]:
import tempfile
from pathlib import Path
from occulus.io import read, write

# Write a synthetic cloud to XYZ, then read it back
tmp = Path(tempfile.mkdtemp())
xyz_path = tmp / 'demo.xyz'
write(cloud, xyz_path)

loaded = read(xyz_path)
print(f'Written : {cloud.n_points:,} points')
print(f'Loaded  : {loaded.n_points:,} points')
print(f'Max abs error: {np.abs(loaded.xyz - cloud.xyz).max():.6f} m')

## 4. Basic Statistics

In [ ]:
from occulus.metrics import compute_cloud_statistics

stats = compute_cloud_statistics(cloud)
print(f'Z min  : {stats.z_min:.3f}')
print(f'Z max  : {stats.z_max:.3f}')
print(f'Z mean : {stats.z_mean:.3f}')
print(f'Z std  : {stats.z_std:.3f}')

## 5. Filtering

In [ ]:
from occulus.filters import (
    voxel_downsample,
    random_downsample,
    statistical_outlier_removal,
    crop,
)

# Add some outliers
outlier_xyz = np.vstack([xyz, rng.uniform(-100, 100, (50, 3))])
noisy = PointCloud(outlier_xyz)

# SOR removal
clean = statistical_outlier_removal(noisy, nb_neighbors=16, std_ratio=2.0)
print(f'Before SOR : {noisy.n_points:,}')
print(f'After SOR  : {clean.n_points:,}  (removed {noisy.n_points - clean.n_points})')

# Voxel downsample
ds = voxel_downsample(cloud, voxel_size=0.1)
print(f'After voxel (0.1): {ds.n_points:,}')

# Crop to bounding box [x_min, y_min, z_min, x_max, y_max, z_max]
cropped = crop(cloud, bbox=np.array([-1, -1, -1, 1, 1, 1]))
print(f'Cropped (±1 cube) : {cropped.n_points:,}')

## 6. Normal Estimation

In [ ]:
from occulus.normals import estimate_normals, orient_normals_to_viewpoint

ds_n = estimate_normals(ds, radius=0.3)
print(f'Has normals: {ds_n.has_normals}')
print(f'Normal shape: {ds_n.normals.shape}')

# Orient toward a viewpoint above the cloud
viewpoint = np.array([0.0, 0.0, 100.0])
oriented = orient_normals_to_viewpoint(ds_n, viewpoint)
# All normals should now have positive Z component
pct_up = (oriented.normals[:, 2] > 0).mean()
print(f'Normals pointing up: {pct_up:.1%}')

---
**Next**: See `02_ground_classification.ipynb` to classify ground vs vegetation returns.